# Clustering Some Layer

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Baseline model weight 확인 : 추후 clustered model의 weight와 비교 예정

In [ ]:
print(model.layers[2].get_weights()[0])

[[[[-0.18906005 -0.03143864  0.10341831 ...  0.00746741  0.0224061
     0.09732284]
   [-0.03801911  0.04806733 -0.05194453 ...  0.13127203  0.02411404
     0.29724228]
   [-0.01280606 -0.06432066 -0.38535205 ... -0.04083483 -0.04700899
     0.02961743]
   ...
   [-0.22288223  0.07884896  0.00818242 ... -0.10550863  0.09420834
     0.04280463]
   [-0.23697877  0.14171639  0.13301988 ...  0.09267303  0.07835148
     0.12182322]
   [-0.08483915  0.01239013 -0.19798319 ... -0.01926844  0.01498523
     0.21646488]]

  [[-0.0635425  -0.04025226  0.01542174 ...  0.00408251  0.15056933
     0.13731803]
   [ 0.14061436 -0.40326047 -0.14663    ... -0.0931893   0.02686312
     0.06496671]
   [ 0.15407771  0.1476861   0.17155364 ... -0.00609724  0.1447987
     0.27336895]
   ...
   [ 0.08966175 -0.10069791 -0.07546911 ... -0.1560787   0.04437348
     0.0237483 ]
   [-0.01001594 -0.09687189  0.17069773 ... -0.05305388  0.21695071
    -0.08687494]
   [ 0.10463184 -0.35434404 -0.29968643 ... -0.0689

## Clustering Some Layer
* **Layer "conv2d_1"**
    * number of cluserts : 8
    * cluster centroids init : K-mean++
* **Layer "dense"**
    * number of cluserts : 16
    * cluster centroids init : K-mean++

In [ ]:
dict_clustering_params = {
    "conv2d_1":{
        'number_of_clusters': 8,
        'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
    },
    "dense":{
        'number_of_clusters': 16,
        'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
    }
}

def apply_clustering_to_dense(layer:keras.layers):
  if layer.name in dict_clustering_params:
    return tfmot.clustering.keras.cluster_weights(layer, **dict_clustering_params[layer.name])
  return layer

clustered_model = keras.models.clone_model(model, clone_function = apply_clustering_to_dense)

## Compile 후 model 확인 : clustering을 위해 metadata가 추가 된 model 확인

In [ ]:
clustered_model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
                        optimizer=keras.optimizers.Adam(learning_rate = 1e-5),  # 작은 leraning rate 사용
                        metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9240      
 eights)                                                         
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                        

## Clustering을 위한 training

In [ ]:
hist_clustered = clustered_model.fit(
    train_images,
    train_labels,
    batch_size=500,
    epochs=1,
    validation_split=0.1
)

108/108 [==============================] - 7s 13ms/step - loss: 0.0067 - accuracy: 0.9976 - val_loss: 0.0415 - val_accuracy: 0.9902


## Accuracy 비교 : baseline model vs clustered model

In [ ]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9904000163078308
Clustered test accuracy: 0.9902999997138977


## Clustered model의 weight 확인 : 8개의 cluster로 weight가 제한 됨

In [ ]:
final_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print(final_model.layers[2].get_weights()[0])

[[[[-0.1716334  -0.01271522  0.0586223  ... -0.01271522 -0.01271522
     0.0586223 ]
   [-0.01271522  0.0586223  -0.08136686 ...  0.18142927  0.0586223
     0.27014416]
   [-0.01271522 -0.08136686 -0.4661655  ... -0.01271522 -0.01271522
     0.0586223 ]
   ...
   [-0.26086503  0.0586223  -0.01271522 ... -0.08136686  0.0586223
     0.0586223 ]
   [-0.26086503  0.18142927  0.18142927 ...  0.0586223   0.0586223
     0.18142927]
   [-0.08136686 -0.01271522 -0.1716334  ... -0.01271522 -0.01271522
     0.18142927]]

  [[-0.08136686 -0.01271522 -0.01271522 ... -0.01271522  0.18142927
     0.18142927]
   [ 0.18142927 -0.4661655  -0.1716334  ... -0.08136686  0.0586223
     0.0586223 ]
   [ 0.18142927  0.18142927  0.18142927 ... -0.01271522  0.18142927
     0.27014416]
   ...
   [ 0.0586223  -0.08136686 -0.08136686 ... -0.1716334   0.0586223
     0.0586223 ]
   [-0.01271522 -0.08136686  0.18142927 ... -0.08136686  0.18142927
    -0.08136686]
   [ 0.0586223  -0.26086503 -0.26086503 ... -0.0813668

In [ ]:
# layer 'conv_2d_1' cluster
set(final_model.layers[2].get_weights()[0].reshape(-1))

{-0.4661655,
 -0.26086503,
 -0.1716334,
 -0.08136686,
 -0.012715223,
 0.0586223,
 0.18142927,
 0.27014416}

In [ ]:
# layer 'dense' cluster
set(final_model.layers[5].get_weights()[0].reshape(-1))

{-0.5107404,
 -0.34333763,
 -0.27013674,
 -0.21551068,
 -0.13040653,
 -0.08193223,
 -0.051475756,
 -0.027959913,
 0.01933069,
 0.06673545,
 0.09654111,
 0.14977096,
 0.20422238,
 0.2497353,
 0.3266134,
 0.49452522}

## 최종 clustered model의 구조 확인
* baseline model과 같음 : memory size에서도 달라진 부분이 없음 (TFMOT clustering 의 한계)

In [ ]:
final_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

## LiteRT 모델로 변환 (Clustering some layer)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
tflite_model = converter.convert()

In [ ]:
tflite_clustering_some_layer_file = save_dir + 'mnist_clustering_some_layer.tflite'
open(tflite_clustering_some_layer_file, 'wb').write(tflite_model)

233596

## 추론 속도 측정

* benchmark_model 설치

In [ ]:
!wget https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
!chmod +x linux_x86-64_benchmark_model

--2025-12-21 14:45:06--  https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.4.207, 74.125.200.207, 74.125.130.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.4.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6685264 (6.4M) [application/octet-stream]
Saving to: ‘linux_x86-64_benchmark_model’

linux_x86-64_benchm 100%[===================>]   6.38M  4.54MB/s    in 1.4s    

2025-12-21 14:45:08 (4.54 MB/s) - ‘linux_x86-64_benchmark_model’ saved [6685264/6685264]



* Baseline model

In [ ]:
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_baseline_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_baseline_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 620.91ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=13353 first=97 curr=31 min=26 max=2498 avg=37.1759 std=25 p5=27 median=36 p95=58

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=29173 first=51 curr=33 min=26 max=1486 avg=34.0265 std=14 p5=27 median=32 p95=49

INFO: Inference timings in us: Init: 620910, First inference: 97, Warmup (avg): 37.1759, Inferenc

* Clustering some layer

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_clustering_some_layer_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_clustering_some_layer.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_clustering_some_layer.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_clustering_some_layer.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 3.44ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=14559 first=80 curr=36 min=26 max=281 avg=34.0926 std=8 p5=27 median=32 p95=49

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=28521 first=69 curr=32 min=25 max=2356 avg=34.8034 std=26 p5=27 median=32 p95=52

INFO: Inference timings in us: Init: 3440, First inference: 80, Warmup (avg): 34

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):

  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

* 압축된 파일 크기 비교

In [ ]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_clustering_model = get_zipped_model_size(final_model)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped pruning model file : {}".format(size_zipped_clustering_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_clustering_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 426993
Size of zipped pruning model file : 49938
ratio : 8.550462573591252
